# Project - Airline AI Assistant

We'll now bring together what we've learned to make an AI Customer Support assistant for an Airline

In [2]:
# imports

import os
import json
from dotenv import load_dotenv
from openai import OpenAI
import gradio as gr
import sqlite3

In [7]:
# Initialization

# load_dotenv(override=True)

# openai_api_key = os.getenv('OPENAI_API_KEY')
# if openai_api_key:
#     print(f"OpenAI API Key exists and begins {openai_api_key[:8]}")
# else:
#     print("OpenAI API Key not set")
    
MODEL = "llama3.2"
ollama = OpenAI(base_url='http://localhost:11434/v1', api_key='ollama')

DB = "prices.db"

In [8]:
system_message = """
You are a helpful assistant for an Airline called FlightAI.
Give short, courteous answers, no more than 1 sentence.
Always be accurate. If you don't know the answer, say so.
"""

In [9]:
def get_ticket_price(city):
    print(f"DATABASE TOOL CALLED: Getting price for {city}", flush=True)
    with sqlite3.connect(DB) as conn:
        cursor = conn.cursor()
        cursor.execute('SELECT price FROM prices WHERE city = ?', (city.lower(),))
        result = cursor.fetchone()
        return f"Ticket price to {city} is ${result[0]}" if result else "No price data available for this city"

In [10]:
get_ticket_price("Paris")

DATABASE TOOL CALLED: Getting price for Paris


'Ticket price to Paris is $899.0'

In [11]:
price_function = {
    "name": "get_ticket_price",
    "description": "Get the price of a return ticket to the destination city.",
    "parameters": {
        "type": "object",
        "properties": {
            "destination_city": {
                "type": "string",
                "description": "The city that the customer wants to travel to",
            },
        },
        "required": ["destination_city"],
        "additionalProperties": False
    }
}
tools = [{"type": "function", "function": price_function}]
tools

[{'type': 'function',
  'function': {'name': 'get_ticket_price',
   'description': 'Get the price of a return ticket to the destination city.',
   'parameters': {'type': 'object',
    'properties': {'destination_city': {'type': 'string',
      'description': 'The city that the customer wants to travel to'}},
    'required': ['destination_city'],
    'additionalProperties': False}}}]

In [13]:

def chat(message, history):
    history = [{"role": h["role"], "content": h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL, messages=messages)
    return response.choices[0].message.content

gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7860
* To create a public link, set `share=True` in `launch()`.


In [15]:
def chat(message, history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history + [{"role": "user", "content": message}]
    response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses = handle_tool_calls(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    
    return response.choices[0].message.content

In [14]:
def handle_tool_calls(message):
    responses = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses

In [16]:
gr.ChatInterface(fn=chat).launch()

* Running on local URL:  http://127.0.0.1:7861
* To create a public link, set `share=True` in `launch()`.


DATABASE TOOL CALLED: Getting price for Tokyo
DATABASE TOOL CALLED: Getting price for Tokyo
DATABASE TOOL CALLED: Getting price for London


## A bit more about what Gradio actually does:

1. Gradio constructs a frontend Svelte app based on our Python description of the UI
2. Gradio starts a server built upon the Starlette web framework listening on a free port that serves this React app
3. Gradio creates backend routes for our callbacks, like chat(), which calls our functions

And of course when Gradio generates the frontend app, it ensures that the the Submit button calls the right backend route.

That's it!

It's simple, and it has a result that feels magical.

# Let's go multi-modal!!

We can use DALL-E-3, the image generation model behind GPT-4o, to make us some images

Let's put this in a function called artist.

### Price alert: each time I generate an image it costs about 4 cents - don't go crazy with images!

In [25]:
# Some imports for handling images

import base64
from io import BytesIO
from PIL import Image
from kokoro import KPipeline
import numpy as np
from IPython.display import Audio, display

In [ ]:
def artist(city):
    image_response = openai.images.generate(
            model="dall-e-3",
            prompt=f"An image representing a vacation in {city}, showing tourist spots and everything unique about {city}, in a vibrant pop-art style",
            size="1024x1024",
            n=1,
            response_format="b64_json",
        )
    image_base64 = image_response.data[0].b64_json
    image_data = base64.b64decode(image_base64)
    return Image.open(BytesIO(image_data))

In [ ]:
image = artist("New York City")
display(image)

In [26]:
# def talker(message):
#     response = openai.audio.speech.create(
#       model="gpt-4o-mini-tts",
#       voice="onyx",    # Also, try replacing onyx with alloy or coral
#       input=message
#     )
#     return response.content


pipeline = KPipeline(lang_code='a')
def talker(message):
  print(f"Đang tạo giọng nói cho: '{message}'...")
    
  # 1. Chạy Kokoro để sinh âm thanh
  generator = pipeline(message, voice='af_heart', speed=1, split_pattern=r'\n+')
  
  # 2. Gộp các đoạn âm thanh lại thành 1 mảng duy nhất
  audio_chunks = [audio for _, _, audio in generator]
  final_audio = np.concatenate(audio_chunks)
  
  # 3. Kích hoạt tự động phát ngay bên trong hàm
  display(Audio(data=final_audio, rate=24000, autoplay=True))
  
  # 4. Trả về dữ liệu gốc (nếu bạn cần dùng mảng âm thanh này cho việc khác sau đó)
  return final_audio

In [28]:
ket_qua = talker("Hello Tommy! This is my new voice running directly inside your function.")

Đang tạo giọng nói cho: 'Hello Tommy! This is my new voice running directly inside your function.'...


## Let's bring this home:

1. A multi-modal AI assistant with image and audio generation
2. Tool callling with database lookup
3. A step towards an Agentic workflow


In [30]:
def chat(history):
    history = [{"role":h["role"], "content":h["content"]} for h in history]
    messages = [{"role": "system", "content": system_message}] + history
    response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)
    cities = []
    image = None

    while response.choices[0].finish_reason=="tool_calls":
        message = response.choices[0].message
        responses, cities = handle_tool_calls_and_return_cities(message)
        messages.append(message)
        messages.extend(responses)
        response = ollama.chat.completions.create(model=MODEL, messages=messages, tools=tools)

    reply = response.choices[0].message.content
    history += [{"role":"assistant", "content":reply}]

    voice = talker(reply)

    # if cities:
    #     image = artist(cities[0])
    
    # return history, voice, image
    return history, voice, image


In [29]:
def handle_tool_calls_and_return_cities(message):
    responses = []
    cities = []
    for tool_call in message.tool_calls:
        if tool_call.function.name == "get_ticket_price":
            arguments = json.loads(tool_call.function.arguments)
            city = arguments.get('destination_city')
            cities.append(city)
            price_details = get_ticket_price(city)
            responses.append({
                "role": "tool",
                "content": price_details,
                "tool_call_id": tool_call.id
            })
    return responses, cities

## The 3 types of Gradio UI

`gr.Interface` is for standard, simple UIs

`gr.ChatInterface` is for standard ChatBot UIs

`gr.Blocks` is for custom UIs where you control the components and the callbacks

In [32]:
# Callbacks (along with the chat() function above)

def put_message_in_chatbot(message, history):
        return "", history + [{"role":"user", "content":message}]

# UI definition

with gr.Blocks() as ui:
    with gr.Row():
        chatbot = gr.Chatbot(height=500)
    with gr.Row():
        audio_output = gr.Audio(autoplay=True)
    with gr.Row():
        message = gr.Textbox(label="Chat with our AI Assistant:")

# Hooking up events to callbacks

    message.submit(put_message_in_chatbot, inputs=[message, chatbot], outputs=[message, chatbot]).then(
        chat, inputs=chatbot, outputs=[chatbot, audio_output]
    )

ui.launch(inbrowser=True, auth=("ed", "bananas"))

* Running on local URL:  http://127.0.0.1:7862
* To create a public link, set `share=True` in `launch()`.


Đang tạo giọng nói cho: '{"name":"get_ticket_price","parameters\":{\"destination_city\":\"your desitination city\"}}'...


c:\Users\titva\anaconda3\envs\llms\Lib\site-packages\gradio\blocks.py:1851: UserWarning: A function (chat) returned too many output values (needed: 2, returned: 3). Ignoring extra values.
    Output components:
        [chatbot, audio]
    Output values returned:
        [[{'role': 'user', 'content': [{'text': 'Hi', 'type': 'text'}]}, {'role': 'assistant', 'content': '{"name":"get_ticket_price","parameters\\":{\\"destination_city\\":\\"your desitination city\\"}}'}], [-1.3671163e-07  1.3679078e-08 -5.8947407e-08 ... -6.7473754e-10
  1.8284819e-10 -1.3271652e-09], None]
  warnings.warn(
Traceback (most recent call last):
  File "c:\Users\titva\anaconda3\envs\llms\Lib\site-packages\gradio\queueing.py", line 856, in process_events
    response = await route_utils.call_process_api(
               ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "c:\Users\titva\anaconda3\envs\llms\Lib\site-packages\gradio\route_utils.py", line 358, in call_process_api
    output = await app.get_blocks().process_a

**TTS**: Kokoro

In [21]:
!pip install kokoro soundfile ipython numpy

  Using cached loguru-0.7.3-py3-none-any.whl.metadata (22 kB)
  Using cached win32_setctime-1.2.0-py3-none-any.whl.metadata (2.4 kB)
  Installing build dependencies: started
  Installing build dependencies: finished with status 'done'
  Getting requirements to build wheel: started
  Getting requirements to build wheel: finished with status 'done'
  Preparing metadata (pyproject.toml): started
  Preparing metadata (pyproject.toml): finished with status 'done'
  Using cached isodate-0.7.2-py3-none-any.whl.metadata (11 kB)
  Using cached rfc3986-1.5.0-py2.py3-none-any.whl.metadata (6.5 kB)
INFO: pip is looking at multiple versions of spacy-curated-transformers to determine which version is compatible with other requirements. This could take a while.
   ---------------------------------------- 0.0/1.0 MB ? eta -:--:--
   ---------------------------------------- 1.0/1.0 MB 8.1 MB/s  0:00:00
   ---------------------------------------- 0.0/3.6 MB ? eta -:--:--
   -------- --------------------

In [22]:
from kokoro import KPipeline
from IPython.display import Audio, display
import numpy as np

# Khởi tạo pipeline cho tiếng Anh - Mỹ
print("Đang khởi tạo model vào RAM...")
pipeline = KPipeline(lang_code='a') 
print("Khởi tạo thành công!")

Đang khởi tạo model vào RAM...


config.json: 0.00B [00:00, ?B/s]

c:\Users\titva\anaconda3\envs\llms\Lib\site-packages\huggingface_hub\file_download.py:138: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\titva\.cache\huggingface\hub\models--hexgrad--Kokoro-82M. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)
c:\Users\titva\anaconda3\envs\llms\Lib\site-packages\torch\nn\modules\rnn.py:123: UserWarning: dropout option adds 

kokoro-v1_0.pth:   0%|          | 0.00/327M [00:00<?, ?B/s]

✔ Download and installation successful
You can now load the package via spacy.load('en_core_web_sm')
⚠ Restart to reload dependencies
If you are in a Jupyter or Colab notebook, you may need to restart Python in
order to load all the package's dependencies. You can do this by selecting the
'Restart kernel' or 'Restart runtime' option.
Khởi tạo thành công!


In [24]:
# 1. Đầu vào (Text & Voice)
text = "Hello Tommy. I am Kokoro. Notice how my voice plays automatically right after the processing is done. Isn't that cool?"
voice = 'af_heart' 

print("Đang chạy luồng G2P và sinh sóng âm...")

# 2. Chạy Pipeline (Kokoro sẽ cắt câu dựa vào dấu chấm xuống dòng)
generator = pipeline(text, voice=voice, speed=1, split_pattern=r'\n+')

# 3. Gộp các đoạn âm thanh nhỏ thành một cục lớn duy nhất
audio_chunks = []
for i, (gs, ps, audio) in enumerate(generator):
    # gs: Graphemes (chữ viết)
    # ps: Phonemes (ngữ âm - IPA)
    # audio: Mảng Numpy sóng âm
    audio_chunks.append(audio)

# Nối các mảng numpy lại với nhau
final_audio = np.concatenate(audio_chunks)

print("Hoàn tất! Bắt đầu phát âm thanh...")

# 4. Hiển thị UI Player và kích hoạt AUTOPLAY
# Tham số autoplay=True chính là thứ bạn cần
display(Audio(data=final_audio, rate=24000, autoplay=True))

Đang chạy luồng G2P và sinh sóng âm...
Hoàn tất! Bắt đầu phát âm thanh...


# Exercises and Business Applications

Add in more tools - perhaps to simulate actually booking a flight. A student has done this and provided their example in the community contributions folder.

Next: take this and apply it to your business. Make a multi-modal AI assistant with tools that could carry out an activity for your work. A customer support assistant? New employee onboarding assistant? So many possibilities! Also, see the week2 end of week Exercise in the separate Notebook.

<table style="margin: 0; text-align: left;">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/thankyou.jpg" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#090;">I have a special request for you</h2>
            <span style="color:#090;">
                My editor tells me that it makes a HUGE difference when students rate this course on Udemy - it's one of the main ways that Udemy decides whether to show it to others. If you're able to take a minute to rate this, I'd be so very grateful! And regardless - always please reach out to me at ed@edwarddonner.com if I can help at any point.
            </span>
        </td>
    </tr>
</table>